In [10]:
# notebooks/06_failure_analysis.ipynb
import sys
import os
import pandas as pd
import numpy as np

# 1. Framework Path Setup
sys.path.append(os.path.abspath('../'))
from src.data_loader import load_validation_splits
from src.scorecard_transformer import ScorecardTransformer

print("=== RUNNING PHASE 6: ADVANCED MRM BOUNDARY & FAILURE FORENSICS ===")

# 2. Ingest Data Splits
df_train_tweaked, df_val_final, df_oot_final, df_scorecard_points = load_validation_splits()

# Use our centralized software transformer layer
transformer = ScorecardTransformer(df_scorecard_points)
print("Calculating validation portfolio scoring layers via transformer asset...")
df_oot_final['credit_score'] = transformer.generate_credit_scores(df_oot_final)

# =====================================================================
# PART A: THE HIGHEST-LOSS RISK DECILE REPORT (MRM REQUIREMENT)
# =====================================================================
print("\n=== HIGH-LOSS RISK DECILE ANALYSIS ===")

# Cut the portfolio into 10 explicit score-based bands (safest to riskiest)
# Lower score = Higher Risk in institutional scorecard frameworks
df_oot_final['score_decile'] = pd.qcut(df_oot_final['credit_score'], q=10, labels=False)

# Group defaults and portfolio exposures
loss_deciles = df_oot_final.groupby('score_decile').agg(
    volume=('bad', 'count'),
    total_defaults=('bad', 'sum'),
    bad_rate=('bad', 'mean')
).reset_index()

# Invert deciles so Decile 1 is the riskiest (lowest scores) and Decile 10 is safest
loss_deciles['score_decile'] = 10 - loss_deciles['score_decile']
loss_deciles = loss_deciles.sort_values(by='score_decile').reset_index(drop=True)

global_defaults = loss_deciles['total_defaults'].sum()
loss_deciles['Error Concentration (% of Total Bads)'] = round((loss_deciles['total_defaults'] / global_defaults) * 100, 2)
loss_deciles['Bad Rate (%)'] = round(loss_deciles['bad_rate'] * 100, 2)

print(loss_deciles[['score_decile', 'volume', 'total_defaults', 'Bad Rate (%)', 'Error Concentration (% of Total Bads)']].to_string(index=False))

# =====================================================================
# PART B: CONFUSION MATRIX BY RISK BAND (MRM REQUIREMENT)
# =====================================================================
print("\n=== CONFUSION MATRIX BY POLICY SCORE BANDS ===")

# Define institutional underwriting cutoff bands
def assign_score_band(score):
    if score < 500: return "1. High Risk (Decline)"
    elif score < 600: return "2. Medium Risk (Refer/Review)"
    return "3. Low Risk (Auto-Approve)"

df_oot_final['policy_band'] = df_oot_final['credit_score'].apply(assign_score_band)

# Pivot into an explicit performance matrix
confusion_matrix_band = pd.crosstab(
    df_oot_final['policy_band'], 
    df_oot_final['bad'], 
    margins=True, 
    margins_name="Total Portfolio"
).rename(columns={0: "Actual Good (Paid)", 1: "Actual Bad (Defaulted)"})

print(confusion_matrix_band.to_string())


# PART C: FAILURE ARCHETYPE ANALYSIS
# ---------------------------------------------------------------------
# Regulatory guidance expects independent validation to investigate
# where the model performs poorly—not only aggregate performance.
#
# This section profiles two operationally important populations:
#   • False Positives:
#       Borrowers assigned relatively safe scores who ultimately default.
#
#   • False Negatives:
#       Borrowers assigned relatively risky scores who ultimately repay.
#
# Rather than treating these as isolated prediction errors, the analysis
# compares their feature distributions against the overall portfolio to
# identify systematic blind spots, potential policy improvements, and
# opportunities for future model redevelopment.
# =====================================================================

# =====================================================================
# PART C: YOUR ARCHETYPE SELECTION LOGIC (PRESERVED & ENHANCED)
# =====================================================================
print("\n=== DEEP FORENSIC FAILURE ARCHETYPE RUN ===")

def profile_false_positives(df_eval, score_col, target_col, top_n=1000):
    """Isolates the highest-scoring (seemingly safest) borrowers who defaulted."""
    defaults_df = df_eval[df_eval[target_col] == 1].copy()
    return defaults_df.sort_values(by=score_col, ascending=False).head(top_n)

def profile_false_negatives(df_eval, score_col, target_col, top_n=1000):
    """Isolates the lowest-scoring (seemingly riskiest) borrowers who repaid successfully."""
    repaid_df = df_eval[df_eval[target_col] == 0].copy()
    return repaid_df.sort_values(by=score_col, ascending=True).head(top_n)

fps_df = profile_false_positives(df_oot_final, 'credit_score', 'bad', top_n=1000)
fns_df = profile_false_negatives(df_oot_final, 'credit_score', 'bad', top_n=1000)

print(f"📊 Isolated {len(fps_df)} False Positives (Model Fooled) and {len(fns_df)} False Negatives (Overly Conservative) for macro testing.")

# Enhanced Forensic Distribution Slices instead of pure global means
print("\n=== RECONSTRUCTED ATTRIBUTE BREAKDOWN PER EXCEPTION POOL ===")
for metric in ['fico_score', 'dti', 'annual_inc']:
    print(f"\nForensic Slicing for Attribute: [{metric.upper()}]")
    summary_metric = pd.DataFrame({
        "Metric Statistic": ["25th Percentile", "Median (50th)", "75th Percentile"],
        "False Positives (High Score Bads)": [round(np.percentile(fps_df[metric].dropna(), p), 2) for p in [25, 50, 75]],
        "False Negatives (Low Score Goods)": [round(np.percentile(fns_df[metric].dropna(), p), p) for p in [25, 50, 75]],
        "Global OOT Baseline": [round(np.percentile(df_oot_final[metric].dropna(), p), 2) for p in [25, 50, 75]]
    })
    print(summary_metric.to_string(index=False))

=== RUNNING PHASE 6: ADVANCED MRM BOUNDARY & FAILURE FORENSICS ===
Calculating validation portfolio scoring layers via transformer asset...

=== HIGH-LOSS RISK DECILE ANALYSIS ===
 score_decile  volume  total_defaults  Bad Rate (%)  Error Concentration (% of Total Bads)
            1    8278           403.0          4.87                                   1.99
            2    8509           876.0         10.29                                   4.33
            3    8936          1214.0         13.59                                   6.00
            4    8891          1553.0         17.47                                   7.67
            5    8384          1790.0         21.35                                   8.85
            6    8547          2094.0         24.50                                  10.35
            7    8360          2381.0         28.48                                  11.77
            8    9507          2966.0         31.20                                  14.66
 